

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [4]:
account = Account.get("muadh")
account

Account(name='muadh', balance=10000.0, strategy='', holdings={}, transactions=[], portfolio_value_time_series=[])

In [5]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

Was not able to use the polygon API due to {"status":"NOT_AUTHORIZED","request_id":"b150432e4bd1efaaad2137f28d2a850f","message":"Attempted to request today's data before end of day. Please upgrade your plan at https://polygon.io/pricing"}; using a random number
Was not able to use the polygon API due to {"status":"NOT_AUTHORIZED","request_id":"bb5c66eb2c48db688fbaa41434474bd8","message":"Attempted to request today's data before end of day. Please upgrade your plan at https://polygon.io/pricing"}; using a random number


'Completed. Latest details:\n{"name": "muadh", "balance": 9879.76, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 40.08, "timestamp": "2026-04-28 01:21:48", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-04-28 01:21:49", 10137.76]], "total_portfolio_value": 10137.76, "total_profit_loss": 137.76000000000022}'

In [6]:
account.report()

Was not able to use the polygon API due to {"status":"NOT_AUTHORIZED","request_id":"ce6bb9663d5a90ac9179c970284d883b","message":"Attempted to request today's data before end of day. Please upgrade your plan at https://polygon.io/pricing"}; using a random number


'{"name": "muadh", "balance": 9879.76, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 40.08, "timestamp": "2026-04-28 01:21:48", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-04-28 01:21:49", 10137.76], ["2026-04-28 01:22:04", 9990.76]], "total_portfolio_value": 9990.76, "total_profit_loss": -9.239999999999782}'

In [7]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 40.08,
  'timestamp': '2026-04-28 01:21:48',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [8]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [9]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [10]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is muadh and my account is under the name muadh. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [12]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Muadh, your current cash balance is $9,879.76. You hold 3 shares of Amazon (AMZN) stock in your account. Is there anything specific you would like to do with your account?

### Now let's build our own MCP Client

In [13]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [14]:
request = "My name is muadh and my account is under the name muadh. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Your current account balance is $9,879.76. Is there anything else you would like to do or know about your account?

In [15]:
context = await read_accounts_resource("muadh")
print(context)

{"name": "muadh", "balance": 9879.76, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 40.08, "timestamp": "2026-04-28 01:21:48", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-04-28 01:21:49", 10137.76], ["2026-04-28 01:22:04", 9990.76], ["2026-04-28 01:25:04", 10152.76]], "total_portfolio_value": 10152.76, "total_profit_loss": 152.76000000000022}


In [ ]:
from accounts import Account
Account.get("muadh").report()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>